In [ ]:


from ultralytics import YOLO
import os
import time
from time import sleep
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
import torch
import numpy as np
import math
from PIL import Image
from transformers import AutoModel
import requests
from gtts import gTTS
from io import BytesIO
from pygame import mixer
import gpiozero
import threading
import RPi. GPIO as GPIO 
import sys
import os
dt_model = YOLO('yolov8n.yaml')
dt_model = YOLO("/home/pi/Downloads/last.pt")

time.sleep(30)

In [ ]:


mymap2 ={
  0: 'person',
  1: 'bicycle',
  2: 'car',
  3: 'motorcycle',
  4: 'airplane',
  5: 'bus',
  6: 'train',
  7: 'truck',
  8: 'boat',
  9: 'traffic light',
  10: 'fire hydrant',
  11: 'stop sign',
  12: 'parking meter',
  13: 'bench',
  14: 'bird',
  15: 'cat',
  16: 'dog',
  17: 'horse',
  18: 'sheep',
  19: 'cow',
  20: 'elephant',
  21: 'bear',
  22: 'zebra',
  23: 'giraffe',
  24: 'backpack',
  25: 'umbrella',
  26: 'handbag',
  27: 'tie',
  28: 'suitcase',
  29: 'frisbee',
  30: 'skis',
  31: 'snowboard',
  32: 'sports_ball',
  33: 'kite',
  34: 'baseball_bat',
  35: 'baseball glove',
  36: 'skateboard',
  37: 'surfboard',
  38: 'tennis_racket',
  39: 'bottle',
  40: 'wine glass',
  41: 'cup',
  42: 'fork',
  43: 'knife',
  44: 'spoon',
  45: 'bowl',
  46: 'banana',
  47: 'apple',
  48: 'sandwich',
  49: 'orange',
  50: 'broccoli',
  51: 'carrot',
  52: 'hot dog',
  53: 'pizza',
  54: 'donut',
  55: 'cake',
  56: 'chair',
  57: 'couch',
  58: 'potted plant',
  59: 'bed',
  60: 'dining table',
  61: 'toilet',
  62: 'tv',
  63: 'laptop',
  64: 'mouse',
  65: 'remote',
  66: 'keyboard',
  67: 'cell phone',
  68: 'microwave',
  69: 'oven',
  70: 'toaster',
  71: 'sink',
  72: 'refrigerator',
  73: 'book',
  74: 'clock',
  75: 'vase',
  76: 'scissors',
  77: 'teddy bear',
  78: 'hair drier',
  79: 'toothbrush'
}

In [ ]:


mymap = {
    0: 'Autorickshaws',
    1: 'Bench',
    2: 'Crosswalk',
    3: 'Firehydrant',
    4: 'Person',
    5: 'Barricade',
    6: 'Garbage box',
    7: 'Pothole',
    8: 'Stairs',
    9: 'Stray animal',
    10: 'green_light',
    11: 'red_light',
    12: 'vehicles'
}

In [ ]:


ss= int('0')
mymap[ss]

In [ ]:


image_processor = AutoImageProcessor.from_pretrained("LiheYoung/depth-anything-small-hf")
depth_model = AutoModelForDepthEstimation.from_pretrained("LiheYoung/depth-anything-small-hf")
threshold_D= 65
path = "/home/pi/"
open_camera_command = "libcamera-hello -t 0"
capture_command = "libcamera-still -n  -t 1  -o /home/pi/test%d.jpg --width 640 --height 640 --shutter 0"


def to_speech(text):
    TEXT =text 
    mixer.init()
    mp3audio = BytesIO()
    myobj = gTTS(text=TEXT, lang='en', slow=False)
    myobj.write_to_fp(mp3audio)
    mp3audio.seek(0)
    mixer.music.load(mp3audio,"mp3")
    mixer.music.play()
    while mixer.music.get_busy():
        pass
    print("we are safe")
    return None


def take_image(i,model):
    # Capture image using libcamera-still
    capture_output = os.system(capture_command % i)
    # Check if capture was successful
    if capture_output == 0:
    # Define the path to the captured image
        image_path = os.path.join(path, f"test{i}.jpg")
        # Wait for a short time to ensure image is saved before processing
        #sleep(0)
        # Perform object detection on the captured image
        image = Image.open(image_path).convert("RGB")
        result = model(image)
        return result,image
    else:
        return None


def get_depth_r(image,image_processor,model):
    inputs = image_processor(images=image, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_depth = outputs.predicted_depth
    
    # interpolate to original size
    prediction = torch.nn.functional.interpolate(
        predicted_depth.unsqueeze(1),
        size=image.size[::-1],
        mode="bicubic",
        align_corners=False,
    )

    # visualize the prediction
    output = prediction.squeeze().cpu().numpy()
    formatted = (output * 255 / np.max(output)).astype("uint8")
    depth = Image.fromarray(formatted)
    #returns the depth image and the depth array
    return formatted,depth

# task executed in a new thread
def task():
    # block for a moment
    GPIO. setmode (GPIO.BCM)
    GPIO. setwarnings (False)
    TRIG = 23
    ECHO = 24
    while True:
        print("Distance Measurement In Progress")
        GPIO.setup (TRIG, GPIO. OUT) 
        GPIO.setup (ECHO, GPIO.IN) 
        GPIO.output (TRIG, False)
        print ("Waiting For Sensor To Settle")
        time.sleep (2)
        GPIO.output (TRIG, True)
        time.sleep (0.00001)
        GPIO.output (TRIG, False) 
        while GPIO.input (ECHO)==0:
            pulse_start = time. time ()
        while GPIO. input (ECHO) ==1:
            pulse_end = time.time()
        pulse_duration = pulse_end - pulse_start
        distance = pulse_duration * 17150
        distance = round (distance,2)
        print( distance )
        if distance < 50.0:
            danger_event.set()
            print('Interrupting main thread now')
            threading.interrupt_main()
            
    return None
    # interrupt the main thread
print("all functions have been loaded")

In [ ]:


# prepare image for the model
#take multiple images in series
#add time

Dangerous_counter = False

img_tkn=7
objectsX=[]
objectsY=[]
objectsCLS=[]
thread = threading.Thread(target = task)
danger_event = threading.Event()
thread.start()
start_text="this is the beginning of our demo Enjoy"
to_speech(start_text)
print("just before for loop")

try:
    for i in range(img_tkn):
        #take image
        if danger_event.is_set():
            danger_event.clear()
            #raise Exception("danger danger hooks in mouth") 
            to_speech("An Object is dangerously close to sensor CARE CARE")
            thread = threading.Thread(target = task)
            thread.start()
                
                
        start_text ="This is the start an image will be taken"
        to_speech(start_text)
        result, image = take_image(i+1,dt_model)
        image.show()
        #get depth map
        depth,Dimage = get_depth_r(image,image_processor,depth_model)
            
        k = len(result)-1
        print(k)
        #print(result)
            
        for box,cls in zip(result[k].boxes.xyxy,result[k].boxes.cls):
            objectsX.append(box[0])
            objectsY.append(box[1])
            objectsCLS.append(cls)
            
        for j in range(len(objectsCLS)):
            relative_distance= depth[math.floor(objectsX[j]),math.floor(objectsY[j])]
                
            if relative_distance < threshold_D :
                print("THIS IS A disasterous OBJECT")
                cls = objectsCLS[j]
                #print(cls)
                cls_num = int(cls.item())
                print("the class number is")
                print(cls_num)
                print(type(cls_num))
                TEXT ='A' + mymap[cls_num] + ' is dangerously close'
                print("the anticipated text is")
                print(TEXT)
                to_speech(TEXT)
                Dangerous_counter= True
                #print("the object is ",object)
            
            else:
                TEXT ="we are safe all good sir"
                #to_speech(TEXT)
                print("we are safe")
        
        if Dangerous_counter:
            Deeztext = "an object is DANGEROUSlY close"
            #to_speech(Deeztext)
        else:
            safetext = "we are safe u are doing fantastic"
            to_speech(safetext)
        objectsX=[]
        objectsY=[]
        objectsCLS=[]    
        Dangerous_counter = False
except:
    print("this is a disaster an exception has occured")
    
    
    
TEXT = "This is the end of our demo   Thank u for listening"
to_speech(TEXT)